In [ ]:
# Enlaces:
"""
1. Sala Especializada en Protección al Consumidor (SPC)
https://www.datosabiertos.gob.pe/dataset/expedientes-presentados-sala-de-protecci%C3%B3n-al-consumidor-indecopi-2018

https://www.datosabiertos.gob.pe/dataset/expedientes-presentados-sala-de-protecci%C3%B3n-al-consumidor-indecopi-2019

https://www.datosabiertos.gob.pe/dataset/expedientes-presentados-sala-de-protecci%C3%B3n-al-consumidor-indecopi-2020

https://www.datosabiertos.gob.pe/dataset/expedientes-presentados-sala-de-protecci%C3%B3n-al-consumidor-indecopi-2021

https://www.datosabiertos.gob.pe/dataset/indecopi-spc-expedientes-presentados


2. Comisiones de Protección al Consumidor (CC1, CC2, CC3)

https://www.datosabiertos.gob.pe/dataset/indecopi-cc1-expedientes-presentados

https://www.datosabiertos.gob.pe/dataset/indecopi-cc2-expedientes-presentados

https://www.datosabiertos.gob.pe/dataset/indecopi-cc3-expedientes-presentados


3. Órgano Resolutivo de Procedimientos Sumarísimos (OPS1, OPS2, OPS3)

https://www.datosabiertos.gob.pe/dataset/indecopi-ops1-expedientes-presentados

https://www.datosabiertos.gob.pe/dataset/indecopi-ops2-expedientes-presentados

https://www.datosabiertos.gob.pe/dataset/indecopi-ops3-expedientes-presentados

"""

# Importación de librerías

In [1]:
import pandas as pd
import glob
import os
import random
import numpy as np

# Carga de los datasets

In [2]:
# 1. CONFIGURACIÓN DE RUTAS
ruta_archivos = './Expedientes Presentados' 
todos_los_archivos = [f for f in glob.glob(os.path.join(ruta_archivos, "*.xls*")) 
                      if not os.path.basename(f).startswith("~$")]
data_list = []

In [3]:
print(f"--- Fase 1: Integración de {len(todos_los_archivos)} fuentes ---")

for archivo in todos_los_archivos:
    try:
        df = pd.read_excel(archivo)
        df.columns = [str(c).upper() for c in df.columns]
        
        temp_df = pd.DataFrame()

        # --- MAPEO DE COLUMNAS ORIGINALES ---
        f_col = [c for c in df.columns if any(x in c for x in ['FECHA', 'FEC_PRE', 'FECHA DE PRESENTACION', 'FECHA PRESENTACION', 'FECHA_PRESENTACION'])]
        if f_col: temp_df['fecha_raw'] = df[f_col[0]]

        t_col = [c for c in df.columns if any(x in c for x in ['TIPO_EXPEDIENTE', 'TIPO_PROCEDIMIENTO', 'TIPO DE EXP', 'VC_TIPO_EXPEDIENTE', 'TIPO EXPEDIENTE','TIPO DE EXP.'])]
        if t_col: temp_df['tipo_expediente'] = df[t_col[0]]

        m_col = [c for c in df.columns if any(x in c for x in ['MATERIA', 'SUB_SECTOR', 'SUBSECTOR', 'RUBRO', 'MATERIAS', 'SUB SECTOR', 'VC_RUBRO'])]
        if m_col: temp_df['materia'] = df[m_col[0]]

        d_col = [c for c in df.columns if any(x in c for x in ['DENUNCIADO', 'AUTORIDAD', 'DENUNCIADOS','DENUNCIADO(S)', 'ADMINISTRADOS'])]
        if d_col: temp_df['denunciado'] = df[d_col[0]]

        if not temp_df.empty:
            data_list.append(temp_df)
            print(f"Agregado: {os.path.basename(archivo)} ({len(temp_df)} filas)")
            
    except Exception as e:
        print(f"Error en {archivo}: {e}")

# Consolidación de base real
df_consolidado = pd.concat(data_list, ignore_index=True)

--- Fase 1: Integración de 18 fuentes ---
Agregado: Expedientes Presentados 2018.xlsx (3400 filas)
Agregado: Expedientes Presentados 2019.xlsx (3125 filas)
Agregado: Expedientes Presentados 2020.xlsx (2026 filas)
Agregado: Expedientes Presentados 2021.xlsx (2750 filas)
Agregado: INDECOPI_CC1_ExpedientesPresentados_2015_0.xlsx (2306 filas)
Agregado: INDECOPI_CC1_ExpedientesPresentados_2016_0.xlsx (2328 filas)
Agregado: INDECOPI_CC2_ExpedientesPresentados_2015.xlsx (2051 filas)
Agregado: INDECOPI_CC2_ExpedientesPresentados_2016.xlsx (2274 filas)
Agregado: INDECOPI_CC3_ExpedientesPresentados_2015.xlsx (115 filas)
Agregado: INDECOPI_CC3_ExpedientesPresentados_2016.xlsx (159 filas)
Agregado: INDECOPI_OPS1_ExpedientesPresentados_2015.xlsx (2363 filas)
Agregado: INDECOPI_OPS1_ExpedientesPresentados_2016.xlsx (2779 filas)
Agregado: INDECOPI_OPS2_ExpedientesPresentados_2015.xlsx (2317 filas)
Agregado: INDECOPI_OPS2_ExpedientesPresentados_2016.xlsx (2383 filas)
Agregado: INDECOPI_OPS3_Expediente

In [4]:
print(f"\n--- Fase 2: Data Augmentation (Asegurando >1M registros) ---")
current_len = len(df_consolidado)
factor_necesario = (1200000 // current_len) + 1
df_masivo = pd.concat([df_consolidado] * factor_necesario, ignore_index=True)
df_masivo = df_masivo.sample(frac=1).reset_index(drop=True)

print(f"Registros base: {current_len} | Factor: x{factor_necesario}")
print(f"Volumen alcanzado: {len(df_masivo)} registros")


--- Fase 2: Data Augmentation (Asegurando >1M registros) ---
Registros base: 43432 | Factor: x28
Volumen alcanzado: 1216096 registros


In [6]:
print(f"\n--- Fase 3: Enriquecimiento Técnico e Inyección de Anomalías ---")

departamentos_peru = [
    'LIMA', 'AREQUIPA', 'CUSCO', 'PIURA', 'CALLAO', 'JUNIN', 'LA LIBERTAD', 
    'LAMBAYEQUE', 'PUNO', 'ANCASH', 'ICA', 'LORETO', 'CAJAMARCA'
]
canales = ['WEB', 'APP_MOVIL', 'TELEFONICO', 'PRESENCIAL']
df_masivo['id_reclamo'] = range(1, len(df_masivo) + 1)
df_masivo['canal'] = [random.choice(canales) for _ in range(len(df_masivo))]

# GENERACIÓN SINTÉTICA DE REGIÓN
df_masivo['region'] = [random.choice(departamentos_peru) for _ in range(len(df_masivo))]

# Horas aleatorias HH:MM:SS
df_masivo['hora'] = [f"{random.randint(0,23):02d}:{random.randint(0,59):02d}:{random.randint(0,59):02d}" for _ in range(len(df_masivo))]

# --- INYECCIÓN DE RED FLAGS ---
# 1. Red Flag de Tiempo: 10,000 registros en el mismo segundo exacto
print("Inyectando Red Flag: 10,000 registros en un solo segundo (10:15:30)...")
idx_t = df_masivo.sample(n=10000).index
df_masivo.loc[idx_t, 'hora'] = "10:15:30"
df_masivo.loc[idx_t, 'canal'] = "WEB"
df_masivo.loc[idx_t, 'region'] = "LIMA" # Simula ataque masivo desde la capital
df_masivo.loc[idx_t, 'materia'] = "RECLAMO_BOT_TIME_ATTACK"

# 2. Red Flag de Texto: 8,000 registros con materia idéntica (Spam)
print("Inyectando Red Flag: Spam masivo de texto repetitivo...")
idx_x = df_masivo.sample(n=8000).index
df_masivo.loc[idx_x, 'materia'] = "SPAM_ADVERTISING_NON_SOLICITED_CONTENT"

# Consolidar Timestamp (YYYY-MM-DD HH:MM:SS)
df_masivo['timestamp'] = df_masivo['fecha_raw'].astype(str).str[:10] + " " + df_masivo['hora']

# Selección de columnas finales
df_final = df_masivo[[
    'id_reclamo', 'timestamp', 'tipo_expediente', 
    'materia', 'denunciado', 'canal', 'region'
]]

# Guardar a CSV
df_final.to_csv('dataset_indecopi_raw_+1M.csv', index=False)

print(f"\n--- PROCESO COMPLETADO ---")
print(f"Archivo generado: dataset_indecopi_raw_1M.csv")
print(f"Total registros: {len(df_final)}")


--- Fase 3: Enriquecimiento Técnico e Inyección de Anomalías ---
Inyectando Red Flag: 10,000 registros en un solo segundo (10:15:30)...
Inyectando Red Flag: Spam masivo de texto repetitivo...

--- PROCESO COMPLETADO ---
Archivo generado: dataset_indecopi_raw_1M.csv
Total registros: 1216096
